In [1]:
import numpy as np
from pathlib import Path

import rtde_control 
import rtde_receive 

In [2]:
ROBOT_IP = "192.168.1.102"
rtde_r = rtde_receive.RTDEReceiveInterface(ROBOT_IP)
rtde_c = rtde_control.RTDEControlInterface(ROBOT_IP)

# --- READ joint positions (radians, 6 joints) ---
joint_q = rtde_r.getActualQ()
print("Joint positions [rad]:", joint_q)

# --- READ Tool Center Point (TCP) pose ---
# Returns [x, y, z, rx, ry, rz] in meters and axis-angle rotation
tcp_pose = rtde_r.getActualTCPPose()
print("TCP pose [m, rad]:", tcp_pose)

Joint positions [rad]: [-0.11495954195131475, -1.6310674152769984, 1.5057480970965784, 0.17921797811474605, 1.2957775592803955, 1.3403759002685547]
TCP pose [m, rad]: [-0.46903675519777616, -0.10792037129766124, 0.5308633392316582, 0.30871202764078165, -1.5949099505077111, -0.04506235325499369]


RTDEReceiveInterface boost system Exception: (asio.misc:2) End of file [asio.misc:2 at /opt/homebrew/anaconda3/include/boost/asio/detail/reactive_socket_recv_op.hpp:134:37 in function 'do_complete']


In [3]:
# Load csv file with x, y, z coordinates
root_dir = Path.cwd().parent
print("Root directory:", root_dir)
csv_path = root_dir / "task2" / "interpolated_rigid_bodies.csv"
waypoints = np.loadtxt(csv_path, delimiter=",", skiprows=1)
print("Waypoints shape:", waypoints.shape)

Root directory: /Users/mathiasnielsen/Documents/Uni - ML/kandidat/2-semester/Simulation based Reinforcement Learning/SiRL/week2
Waypoints shape: (1000, 3)


In [ ]:
#Scale waypoints to fit within robot workspace
waypoints *= 0.1

# Read TCP pose
tcp_pose = rtde_r.getActualTCPPose()

T0 = waypoints[0]

tcp_pos = np.array(tcp_pose[:3])  # Extract x, y, z from TCP pose
tcp_orientation = tcp_pose[3:]  # Extract orientation (rx, ry, rz)

translation_vector = tcp_pos - T0

print("translation vector: ", translation_vector)



# Apply translation to all waypoints
translated_waypoints = waypoints + translation_vector


translation vector:  [-0.34851225 -0.41028768  0.49930188]


In [ ]:
# ── 5. Replay trajectory with moveL path + blending ──────────────────────────
speed        = 0.1    # m/s
acceleration = 0.1    # m/s²
blend        = 0.001   # metres — smooth blending between waypoints (0 = stop at each)



##Move to initial pos
rtde_c = rtde_control.RTDEControlInterface(ROBOT_IP)
rtde_c.moveL(translated_waypoints[0], speed,acceleration, blend)

In [6]:
path = []
print("TCP orientation (rx, ry, rz):", tcp_orientation)
for i, point in enumerate(translated_waypoints):
    # Last waypoint must have blend = 0 (no blending at the end)
    b = 0.0 if i == len(translated_waypoints) - 1 else blend
    point = list(point) + tcp_orientation + [speed, acceleration, b]
    path.append(point)

rtde_c.moveL(path)   # executes the full path in one call with smooth blending

# ── 6. Clean up ──────────────────────────────────────────────────────────────
rtde_c.stopScript()

TCP orientation (rx, ry, rz): [0.30871202764078165, -1.5949099505077111, -0.04506235325499369]


RTDEControlInterface: Could not receive data from robot...
RTDEControlInterface Exception: Operation canceled [system:89 at /opt/homebrew/anaconda3/include/boost/asio/detail/reactive_socket_recv_op.hpp:134:37 in function 'do_complete']
RTDEControlInterface: Robot is disconnected, reconnecting...
RTDEControlInterface Exception: Timeout connecting to UR dashboard server.


Reconnecting...
